# Model Evaluation - Maternal Health Risk Prediction

## Objective

This notebook evaluates the performance of the developed machine learning models for maternal health risk prediction.

The following models were trained and saved during the model development stage:

1. Logistic Regression (Baseline Model)
2. Decision Tree
3. Random Forest
4. Tuned Random Forest

The models will be compared using evaluation metrics including:

- Accuracy
- Precision
- Recall
- F1-score
- Confusion Matrix
- ROC-AUC Score
- Feature Importance

The goal is to identify the most effective model for predicting maternal health risk levels.

In [4]:
# Data manipulation
import polars as pl

# Numerical operations
import numpy as np

# Visualization
import matplotlib.pyplot as plt

# Model evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

# Model loading
import joblib

from pathlib import Path

In [11]:
# Load trained models

logistic_model = joblib.load(
    "../models/logistic_regression.pkl"
)


decision_tree_model = joblib.load(
    "../models/decision_tree.pkl"
)


random_forest_model = joblib.load(
    "../models/random_forest.pkl"
)


tuned_random_forest_model = joblib.load(
    "../models/tuned_random_forest.pkl"
)


print("All models loaded successfully!")

All models loaded successfully!


In [7]:
# Load test data
logistic_test = pl.read_csv(
    "../data/processed/logistic_test_data.csv"
)
logistic_test.head()

column_0,column_1,column_2,column_3,column_4,column_5,column_6,column_7,column_8,column_9,column_10,column_11,column_12,column_13,risk_level
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64
-0.008972,0.368952,0.420295,-1.21172,0.185038,-0.893168,0.210305,0.871018,0.254031,-0.438478,2.984,-1.292203,1.0,0.0,0
0.172154,-0.972692,-0.936274,1.085106,0.208352,-0.641987,-0.787675,0.428334,0.667752,-0.438478,-0.335121,-0.846021,1.0,0.0,1
-1.27685,-0.972692,-0.936274,-0.630596,-0.257927,-1.646713,0.210305,1.01858,-0.32518,-0.438478,-0.335121,-0.399839,0.0,0.0,1
1.258907,1.710596,1.776865,-2.11108,-1.843275,-0.830373,1.208284,0.379147,-0.407924,-0.438478,-0.335121,-0.846021,1.0,0.0,2
-0.371223,-0.525477,-0.484084,-2.346297,-0.607636,-0.139623,-1.332028,-0.653782,1.246963,-0.438478,-0.335121,-1.738385,1.0,0.0,2


In [13]:
X_test_logistic = logistic_test.drop(
    "risk_level"
)


y_test_logistic = logistic_test["risk_level"]

In [8]:
tree_test = pl.read_csv(
    "../data/processed/tree_models_test_data.csv"
)
tree_test.head()

column_0,column_1,column_2,column_3,column_4,column_5,column_6,column_7,column_8,column_9,column_10,column_11,column_12,column_13,risk_level
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64
26.0,4.0,3.0,18.6,24.5,102.0,75.0,12.1,88.0,0.0,1.0,1.0,1.0,0.0,0
27.0,1.0,0.0,35.2,24.6,106.0,64.0,11.2,93.0,0.0,0.0,2.0,1.0,0.0,1
19.0,1.0,0.0,22.8,22.6,90.0,75.0,12.4,81.0,0.0,0.0,3.0,0.0,0.0,1
33.0,7.0,6.0,12.1,15.8,103.0,86.0,11.1,80.0,0.0,0.0,2.0,1.0,0.0,2
24.0,2.0,1.0,10.4,21.1,114.0,58.0,9.0,100.0,0.0,0.0,0.0,1.0,0.0,2


In [14]:
X_test_tree = tree_test.drop(
    "risk_level"
)


y_test_tree = tree_test["risk_level"]

In [15]:
# Logistic Regression

X_test_logistic_array = X_test_logistic.to_numpy()

y_test_logistic_array = y_test_logistic.to_numpy()



# Tree models

X_test_tree_array = X_test_tree.to_numpy()

y_test_tree_array = y_test_tree.to_numpy()

# Generate Predictions

In [16]:
predictions = {}


# Logistic Regression prediction

predictions["Logistic Regression"] = (
    logistic_model.predict(
        X_test_logistic_array
    )
)



# Decision Tree prediction

predictions["Decision Tree"] = (
    decision_tree_model.predict(
        X_test_tree_array
    )
)



# Random Forest prediction

predictions["Random Forest"] = (
    random_forest_model.predict(
        X_test_tree_array
    )
)



# Tuned Random Forest prediction

predictions["Tuned Random Forest"] = (
    tuned_random_forest_model.predict(
        X_test_tree_array
    )
)


print("Predictions generated successfully!")

Predictions generated successfully!


# Evaluate All Models

In [17]:
results = []


# Store corresponding true labels

true_labels = {

    "Logistic Regression": y_test_logistic_array,

    "Decision Tree": y_test_tree_array,

    "Random Forest": y_test_tree_array,

    "Tuned Random Forest": y_test_tree_array
}



for model_name, prediction in predictions.items():

    y_true = true_labels[model_name]


    results.append({

        "Model": model_name,

        "Accuracy": accuracy_score(
            y_true,
            prediction
        ),

        "Precision": precision_score(
            y_true,
            prediction,
            average="weighted"
        ),

        "Recall": recall_score(
            y_true,
            prediction,
            average="weighted"
        ),

        "F1 Score": f1_score(
            y_true,
            prediction,
            average="weighted"
        )

    })


results_df = pl.DataFrame(results)


results_df.sort(
    "Recall",
    descending=True
)

Model,Accuracy,Precision,Recall,F1 Score
str,f64,f64,f64,f64
"""Random Forest""",0.940667,0.943187,0.940667,0.940323
"""Tuned Random Forest""",0.940333,0.943098,0.940333,0.93997
"""Decision Tree""",0.896167,0.896517,0.896167,0.896321
"""Logistic Regression""",0.780667,0.786488,0.780667,0.781104


## Observation

The performance of the four developed models was compared using Accuracy, Precision, Recall, and F1-score.

The results show that the ensemble-based models performed better than the baseline Logistic Regression model.

- **Random Forest achieved the best overall performance**, with an accuracy of approximately **94.07%**, recall of **94.07%**, and F1-score of **94.03%**. This indicates that the model was able to correctly identify most maternal health risk categories while maintaining a good balance between precision and recall.

- **Tuned Random Forest produced similar performance** to the original Random Forest model, achieving approximately **94.03% accuracy** and **94.00% recall**. Although hyperparameter tuning improved model optimization, it did not produce a significant performance improvement compared to the default Random Forest model.

- **Decision Tree achieved good performance**, with approximately **89.62% accuracy and recall**. However, it performed lower than the Random Forest models, suggesting that combining multiple decision trees improved prediction stability and generalization.

- **Logistic Regression (Baseline Model) recorded the lowest performance**, with approximately **78.07% accuracy and recall**. This indicates that the relationship between maternal health features and risk levels may be complex and better captured by non-linear models such as Random Forest.

Overall, the results demonstrate that tree-based ensemble models are more suitable for this maternal health risk prediction task. The Random Forest model achieved the highest overall performance and provides a strong candidate for the final prediction model.

However, final model selection will also consider the **High-Risk Recall score**, because correctly identifying high-risk pregnancies is the priority in maternal healthcare. A model with higher high-risk recall reduces the possibility of missing mothers who require additional medical attention.

# High-Risk Recall Evaluation

In [23]:
from sklearn.metrics import classification_report

# High-risk class label
high_risk_label = 0


for model_name, prediction in predictions.items():

    y_true = true_labels[model_name]


    report = classification_report(
        y_true,
        prediction,
        output_dict=True
    )


    high_risk_recall = report[
        str(high_risk_label)
    ]["recall"]


    print(
        model_name,
        "High Risk Recall:",
        round(high_risk_recall, 3)
    )

Logistic Regression High Risk Recall: 0.718
Decision Tree High Risk Recall: 0.879
Random Forest High Risk Recall: 0.857
Tuned Random Forest High Risk Recall: 0.853


## Observation: High-Risk Recall Evaluation

High-risk recall was evaluated separately because correctly identifying mothers in the high-risk category is the primary objective of this project.

The results show that:

- **Decision Tree achieved the highest high-risk recall of 87.9%**, meaning it successfully identified the largest proportion of actual high-risk pregnancies among all evaluated models. This reduces the possibility of missing mothers who may require additional medical attention.

- **Random Forest achieved a high-risk recall of 85.7%**, which was slightly lower than Decision Tree. However, it achieved the highest overall accuracy, precision, recall, and F1-score, showing stronger general performance across all risk categories.

- **Tuned Random Forest recorded a high-risk recall of 85.3%**, which was very close to the standard Random Forest performance. The tuning process did not significantly improve the detection of high-risk cases.

- **Logistic Regression recorded the lowest high-risk recall of 71.8%**, indicating that the baseline model missed a higher number of actual high-risk pregnancy cases compared to the tree-based models.

For maternal health applications, high-risk recall is a critical consideration because false negatives (predicting a high-risk pregnancy as lower risk) can have serious consequences.

Although Random Forest provided the best overall predictive performance, Decision Tree demonstrated the strongest ability to detect high-risk pregnancies. Therefore, final model selection should consider the trade-off between overall performance and the ability to identify high-risk cases.

# Classification Report for Best Model

In [24]:
best_model = (
    results_df
    .sort(
        "Recall",
        descending=True
    )
    .row(0)[0]
)


print(
    "Best Model Based on Recall:",
    best_model
)

Best Model Based on Recall: Random Forest


In [25]:
print(
    classification_report(
        true_labels[best_model],
        predictions[best_model]
    )
)

              precision    recall  f1-score   support

           0       0.99      0.86      0.92      1468
           1       0.95      0.99      0.97      2420
           2       0.89      0.94      0.92      2112

    accuracy                           0.94      6000
   macro avg       0.95      0.93      0.94      6000
weighted avg       0.94      0.94      0.94      6000



## Observation: Classification Report (Random Forest)

The classification report provides detailed performance evaluation of the Random Forest model across the three maternal health risk categories.

- The model achieved an overall accuracy of **94%**, showing strong predictive performance across the test dataset.

- For the **High Risk class (Class 0)**, the model achieved:
  - **Precision: 99%**
  - **Recall: 86%**
  - **F1-score: 92%**

  The high precision indicates that when the model predicts a pregnancy as high risk, it is almost always correct. However, the recall of 86% means that the model successfully identified 86% of actual high-risk pregnancies, while some high-risk cases were still missed.

- For **Class 1**, the model achieved the highest recall of **99%**, indicating excellent identification of this category.

- For **Class 2**, the model achieved a recall of **94%**, showing strong performance in identifying this risk category.

Overall, the Random Forest model demonstrated strong and balanced performance across all classes. For maternal health prediction, the high-risk recall of **86%** is particularly important because it represents the model's ability to identify mothers who may require additional medical attention.

Although some high-risk cases were missed, the model provides a strong foundation for a maternal health risk screening support system.

# Confusion Matrix

In [26]:
cm = confusion_matrix(
    true_labels[best_model],
    predictions[best_model]
)


cm_df = pl.DataFrame(cm)

cm_df

column_0,column_1,column_2
i64,i64,i64
1258,3,207
0,2392,28
7,111,1994


## Observation: Confusion Matrix (Random Forest)

The confusion matrix provides insight into how the Random Forest model classified each maternal health risk category.

The diagonal values represent correct predictions, while the off-diagonal values represent incorrect classifications.

- For the **High Risk class (Class 0)**:
  - The model correctly identified **1,258 out of 1,468 high-risk cases**.
  - It incorrectly classified **210 high-risk cases** as lower-risk categories, with most being predicted as Class 2.
  - This resulted in a high-risk recall of approximately **85.7%**, meaning the model successfully detected the majority of high-risk pregnancies.

- For **Class 1**:
  - The model correctly identified **2,392 out of 2,420 cases**, showing excellent performance with very few misclassifications.

- For **Class 2**:
  - The model correctly identified **1,994 out of 2,112 cases**, demonstrating strong classification performance.

Overall, the confusion matrix shows that the Random Forest model performs well across all maternal health risk categories. However, the main area for improvement is reducing false negatives in the High Risk class, as missed high-risk pregnancies represent the most important error in a healthcare screening system.

# ROC-AUC Score
## Logistic Regression

In [27]:
logistic_probability = logistic_model.predict_proba(
    X_test_logistic_array
)


logistic_auc = roc_auc_score(
    y_test_logistic_array,
    logistic_probability,
    multi_class="ovr"
)


print(
    "Logistic Regression ROC-AUC:",
    round(logistic_auc,3)
)

Logistic Regression ROC-AUC: 0.913


## Tuned Random Forest

In [28]:
rf_probability = tuned_random_forest_model.predict_proba(
    X_test_tree_array
)


rf_auc = roc_auc_score(
    y_test_tree_array,
    rf_probability,
    multi_class="ovr"
)


print(
    "Tuned Random Forest ROC-AUC:",
    round(rf_auc,3)
)

Tuned Random Forest ROC-AUC: 0.984


## Observation: ROC-AUC Evaluation

The ROC-AUC score was used to evaluate the models' ability to distinguish between different maternal health risk categories.

- **Logistic Regression achieved a ROC-AUC score of 0.913**, indicating that the baseline model has a good ability to separate risk categories. However, its performance is lower compared to the tree-based ensemble model.

- **Tuned Random Forest achieved a ROC-AUC score of 0.984**, demonstrating excellent discriminative ability. This indicates that the tuned model can effectively distinguish between low-risk, moderate-risk, and high-risk pregnancy categories.

The higher ROC-AUC performance of Tuned Random Forest suggests that the model captures complex non-linear relationships between maternal health features better than Logistic Regression.

Overall, the ROC-AUC results support the strong performance of Random Forest-based models for maternal health risk prediction.

# Feature Importance (Tuned Random Forest)

In [32]:
import polars as pl

tree_test = pl.read_csv(
    "../data/processed/tree_models_test_data.csv"
)

tree_test.head()

column_0,column_1,column_2,column_3,column_4,column_5,column_6,column_7,column_8,column_9,column_10,column_11,column_12,column_13,risk_level
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,i64
26.0,4.0,3.0,18.6,24.5,102.0,75.0,12.1,88.0,0.0,1.0,1.0,1.0,0.0,0
27.0,1.0,0.0,35.2,24.6,106.0,64.0,11.2,93.0,0.0,0.0,2.0,1.0,0.0,1
19.0,1.0,0.0,22.8,22.6,90.0,75.0,12.4,81.0,0.0,0.0,3.0,0.0,0.0,1
33.0,7.0,6.0,12.1,15.8,103.0,86.0,11.1,80.0,0.0,0.0,2.0,1.0,0.0,2
24.0,2.0,1.0,10.4,21.1,114.0,58.0,9.0,100.0,0.0,0.0,0.0,1.0,0.0,2


In [34]:
type(tree_test)

polars.dataframe.frame.DataFrame

In [35]:
tree_test.shape

(6000, 15)

In [36]:
tree_test.columns

['column_0',
 'column_1',
 'column_2',
 'column_3',
 'column_4',
 'column_5',
 'column_6',
 'column_7',
 'column_8',
 'column_9',
 'column_10',
 'column_11',
 'column_12',
 'column_13',
 'risk_level']

In [38]:
processed_df = pl.read_csv(
    "../data/processed/maternal_health_processed.csv"
)

In [39]:
processed_df.columns

['age_years',
 'gravidity',
 'parity',
 'gestational_age_weeks',
 'bmi_pre_pregnancy',
 'systolic_bp_mmhg',
 'diastolic_bp_mmhg',
 'hemoglobin_gdl',
 'fasting_glucose_mgdl',
 'proteinuria',
 'hiv_status',
 'anc_visits',
 'delivery_mode',
 'pregnancy_outcome',
 'risk_level']

In [40]:
feature_names = (
    processed_df
    .drop("risk_level")
    .columns
)

feature_names

['age_years',
 'gravidity',
 'parity',
 'gestational_age_weeks',
 'bmi_pre_pregnancy',
 'systolic_bp_mmhg',
 'diastolic_bp_mmhg',
 'hemoglobin_gdl',
 'fasting_glucose_mgdl',
 'proteinuria',
 'hiv_status',
 'anc_visits',
 'delivery_mode',
 'pregnancy_outcome']

In [41]:
feature_importance = pl.DataFrame(
    {
        "Feature": feature_names,
        "Importance": tuned_random_forest_model.feature_importances_
    }
)


feature_importance = feature_importance.sort(
    "Importance",
    descending=True
)


feature_importance

Feature,Importance
str,f64
"""hemoglobin_gdl""",0.377334
"""hiv_status""",0.182164
"""fasting_glucose_mgdl""",0.089425
"""age_years""",0.073159
"""systolic_bp_mmhg""",0.071146
…,…
"""bmi_pre_pregnancy""",0.021601
"""gestational_age_weeks""",0.0157
"""pregnancy_outcome""",0.01188


## Observation: Feature Importance Analysis

Feature importance analysis was performed using the Tuned Random Forest model to identify the maternal health factors that contributed most to the model's predictions.

The results show that the model relied heavily on a few key features:

- **Hemoglobin level (`hemoglobin_gdl`) was the most influential feature**, contributing approximately **37.7%** of the model's decision-making. This highlights the importance of maternal blood health indicators, as low hemoglobin levels may be associated with anemia and increased pregnancy-related risks.

- **HIV status (`hiv_status`) was the second most important feature**, contributing approximately **18.2%**. This indicates that the model identified HIV status as an important factor in distinguishing maternal health risk categories.

- **Fasting glucose level (`fasting_glucose_mgdl`) contributed approximately **8.9%**, showing the relevance of blood sugar regulation in maternal risk prediction.

- Other influential features included **age (`age_years`)** and **systolic blood pressure (`systolic_bp_mmhg`)**, contributing approximately **7.3%** and **7.1%** respectively. These factors are commonly associated with pregnancy complications and maternal health outcomes.

The least influential features in the model were **delivery mode**, **ANC visits**, and **pregnancy outcome**, suggesting that they contributed less to the model's classification decisions compared to the major clinical indicators.

Overall, the feature importance analysis shows that the Tuned Random Forest model identified medically relevant maternal health factors when predicting pregnancy risk levels. These insights improve model interpretability and provide a better understanding of the factors influencing maternal health risk predictions.

## Note on Model Interpretation

Feature importance indicates which variables influenced the model's predictions most strongly. However, feature importance does not imply causation. Further clinical validation would be required before using these findings for medical decision-making.

# Final Conclusion

This project developed and evaluated four machine learning models for maternal health risk prediction:

- Logistic Regression (Baseline Model)
- Decision Tree
- Random Forest
- Tuned Random Forest

The models were evaluated using multiple performance metrics, including Accuracy, Precision, Recall, F1-score, Confusion Matrix, ROC-AUC, and Feature Importance.

The evaluation results showed that the tree-based models significantly outperformed the Logistic Regression baseline model. Random Forest achieved the strongest overall classification performance, with approximately **94% accuracy, precision, recall, and F1-score**, demonstrating its ability to effectively classify maternal health risk levels.

Although Decision Tree achieved the highest **High-Risk Recall (87.9%)**, Random Forest provided a better balance between detecting high-risk cases and maintaining strong performance across all risk categories, achieving a High-Risk Recall of approximately **85.7%**.

The ROC-AUC evaluation further supported the effectiveness of tree-based ensemble models, with the Tuned Random Forest achieving an excellent ROC-AUC score of **0.984**, indicating strong ability to distinguish between different maternal health risk categories.

Feature importance analysis from the Tuned Random Forest model revealed that important maternal health indicators such as **hemoglobin level, HIV status, fasting glucose level, age, and systolic blood pressure** contributed significantly to risk prediction. These findings highlight the importance of clinical and demographic factors in identifying maternal health risks.

Based on the overall evaluation, **Random Forest was selected as the final model** because it achieved the best balance between predictive performance, reliability, and the ability to identify different maternal health risk categories.

However, this model should be considered a decision-support tool rather than a replacement for medical professionals. Further validation with real-world clinical data and expert review would be required before deployment in healthcare settings.

In [42]:
import joblib
import numpy as np

In [43]:
model = joblib.load(
    "../models/random_forest.pkl"
)

In [44]:
model

,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstr

In [ ]:
import numpy as np
import joblib

# Load the saved Random Forest model
model = joblib.load("../models/random_forest.pkl")


# Sample maternal health data
# Features are in the exact order used during training

sample_data = np.array([[
    28,      # age_years
    2,       # gravidity
    1,       # parity
    32,      # gestational_age_weeks
    24.5,    # bmi_pre_pregnancy
    120,     # systolic_bp_mmhg
    80,      # diastolic_bp_mmhg
    12.5,    # hemoglobin_gdl
    90,      # fasting_glucose_mgdl
    0,       # proteinuria
    0,       # hiv_status: Negative
    5,       # anc_visits
    1,       # delivery_mode: Vaginal
    0        # pregnancy_outcome: Live birth
]])


# Make prediction
prediction = model.predict(sample_data)


# Convert encoded prediction to risk category
risk_mapping = {
    0: "High Risk",
    1: "Low Risk",
    2: "Mid Risk"
}


print("Predicted Risk:", risk_mapping[prediction[0]])

Predicted Risk: Low Risk


In [5]:
model.get_params()

{'bootstrap': True,
 'ccp_alpha': 0.0,
 'class_weight': None,
 'criterion': 'gini',
 'max_depth': None,
 'max_features': 'sqrt',
 'max_leaf_nodes': None,
 'max_samples': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'n_estimators': 100,
 'n_jobs': None,
 'oob_score': False,
 'random_state': 42,
 'verbose': 0,
 'warm_start': False}

In [7]:
import polars as pl
model_parameters = pl.DataFrame(
    {
        "Parameter": list(model.get_params().keys()),
        "Value": [
            str(value)
            for value in model.get_params().values()
        ]
    }
)

model_parameters

Parameter,Value
str,str
"""bootstrap""","""True"""
"""ccp_alpha""","""0.0"""
"""class_weight""","""None"""
"""criterion""","""gini"""
"""max_depth""","""None"""
…,…
"""n_jobs""","""None"""
"""oob_score""","""False"""
"""random_state""","""42"""


In [8]:
print(model)

RandomForestClassifier(random_state=42)


In [1]:
import joblib

model = joblib.load("../models/random_forest.pkl")

print("Model type:", type(model))
print("Number of features:", model.n_features_in_)

if hasattr(model, "feature_names_in_"):
    print("Features:")
    for feature in model.feature_names_in_:
        print("-", feature)

Model type: <class 'sklearn.ensemble._forest.RandomForestClassifier'>
Number of features: 14


In [2]:
# Check the exact feature order used by the saved model

if hasattr(model, "feature_names_in_"):
    print("Feature order:")
    for i, feature in enumerate(model.feature_names_in_, start=1):
        print(f"{i}. {feature}")
else:
    print("The saved model does not contain feature names.")

The saved model does not contain feature names.


In [5]:
# ============================================
# Pregnancy Risk Prediction Function
# ============================================

import joblib
import numpy as np

# Load the trained Random Forest model
model = joblib.load("../models/random_forest.pkl")

# Risk category mapping
risk_mapping = {
    0: "High Risk",
    1: "Low Risk",
    2: "Mid Risk"
}


def predict_pregnancy_risk(
    age_years,
    gravidity,
    parity,
    gestational_age_weeks,
    bmi_pre_pregnancy,
    systolic_bp_mmhg,
    diastolic_bp_mmhg,
    hemoglobin_gdl,
    fasting_glucose_mgdl,
    proteinuria,
    hiv_status,
    anc_visits,
    delivery_mode,
    pregnancy_outcome
):
    """
    Predict pregnancy risk using the trained Random Forest model.
    """

    # Arrange features in the exact order used during training
    input_data = np.array([[
        age_years,
        gravidity,
        parity,
        gestational_age_weeks,
        bmi_pre_pregnancy,
        systolic_bp_mmhg,
        diastolic_bp_mmhg,
        hemoglobin_gdl,
        fasting_glucose_mgdl,
        proteinuria,
        hiv_status,
        anc_visits,
        delivery_mode,
        pregnancy_outcome
    ]])

    # Make prediction
    prediction = model.predict(input_data)[0]

    # Get probabilities
    probabilities = model.predict_proba(input_data)[0]

    # Convert numerical prediction to risk category
    risk_level = risk_mapping[prediction]

    return {
        "risk_level": risk_level,
        "prediction_class": int(prediction),
        "probabilities": probabilities.tolist()
    }

In [6]:
result = predict_pregnancy_risk(
    age_years=28,
    gravidity=2,
    parity=1,
    gestational_age_weeks=32,
    bmi_pre_pregnancy=24.5,
    systolic_bp_mmhg=120,
    diastolic_bp_mmhg=80,
    hemoglobin_gdl=12.5,
    fasting_glucose_mgdl=90,
    proteinuria=0,
    hiv_status=0,
    anc_visits=5,
    delivery_mode=1,
    pregnancy_outcome=0
)

print(result)

{'risk_level': 'Low Risk', 'prediction_class': 1, 'probabilities': [0.0, 0.98, 0.02]}
